# Notebook 1: Data Collection & Exploratory Data Analysis

**Business problem:** Brands invest millions in influencer marketing with no systematic way to measure what works.
This notebook collects YouTube data across 30 creators (Tech + Lifestyle, Nano/Micro/Macro) and explores the raw patterns.

**Data sources:** YouTube Data API v3 — channel metadata, video metadata, top-level comments
**Period:** July 2024 – July 2025 (12 months)


In [ ]:
# Install dependencies
# !pip install google-api-python-client pandas tqdm python-dateutil isodate matplotlib seaborn

import os
import json
import time
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import isodate
from datetime import datetime, timedelta
from tqdm import tqdm
from googleapiclient.discovery import build
from dotenv import load_dotenv

load_dotenv()  # loads API key from .env file
API_KEY = os.getenv('YOUTUBE_API_KEY')

sns.set(style='whitegrid')
print('Libraries loaded')

In [ ]:
# ── DATA LOADING ──
# Update these paths to your local data folder
DATA_DIR = '../data/'

channels_df = pd.read_csv(DATA_DIR + 'channels_data_2024_2025.csv')
videos_df   = pd.read_csv(DATA_DIR + 'videos_data_2024_2025.csv')
comments_df = pd.read_csv(DATA_DIR + 'comments_data_2024_2025.csv')

print('Channels:', channels_df.shape)
print('Videos:  ', videos_df.shape)
print('Comments:', comments_df.shape)

In [ ]:

# Shapes and columns
print("Channels:", channels_df.shape)
print(channels_df.columns)
print("Videos:", videos_df.shape)
print(videos_df.columns)
print("Comments:", comments_df.shape)
print(comments_df.columns)

# Display the first few rows to confirm
print(channels_df.head())
print(videos_df.head())
print(comments_df.head())

In [ ]:
print(channels_df[['name', 'genre', 'sizeCategory', 'videosInLastYear']])
print("\nSummary stats:")
print(channels_df['videosInLastYear'].describe())
print(channels_df.columns)

In [ ]:
plt.figure(figsize=(10,5))
plt.hist(channels_df['videosInLastYear'], bins=15, color='skyblue', edgecolor='black')
plt.xlabel('Videos in Last Year')
plt.ylabel('Number of Creators')
plt.title('Distribution of Creator Uploads')
plt.show()

In [ ]:
#Flagging Shorts in the dataframe

# Add a column 'is_short' (True if duration < 60 seconds)
videos_df['is_short'] = videos_df['duration_sec'] <= 60

# Quick check: how many Shorts total?
print(videos_df['is_short'].value_counts())
print("Total Shorts videos:", videos_df['is_short'].sum())
print("Total traditional videos:", (~videos_df['is_short']).sum())

In [ ]:
# Aggregate Shorts and Traditional by creator
summary = videos_df.groupby('creatorName')['is_short'].value_counts().unstack(fill_value=0)
summary.columns = ['Traditional', 'Shorts'] if False in summary.columns else summary.columns
summary = summary.rename(columns={False: 'Traditional', True: 'Shorts'})
summary['Total'] = summary['Traditional'] + summary['Shorts']
summary['%Shorts'] = 100 * summary['Shorts'] / summary['Total']
print(summary.sort_values('Shorts', ascending=False))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -- Setup --
sns.set(style="whitegrid")
plt.figure(figsize=(16, 6))  # Wider for long names

# -- Calculate Shorts % --
creator_counts = videos_df.groupby(['creatorName', 'is_short']).size().unstack(fill_value=0)
creator_counts['Total'] = creator_counts.sum(axis=1)
creator_counts['Shorts %'] = 100 * creator_counts[True] / creator_counts['Total']
creator_counts = creator_counts.sort_values('Shorts %', ascending=False)

# -- Plot --
bars = plt.bar(
    creator_counts.index,
    creator_counts['Shorts %'],
    color=sns.color_palette("pastel", len(creator_counts))
)

# -- Add Labels --
for bar in bars:
    height = bar.get_height()
    if height > 10:
      y = height / 2
      va = 'center'
    else:
      y = height + 2
      va = 'bottom'
    plt.text(
          bar.get_x() + bar.get_width() / 2,
            height / 2,  # Middle of the bar
            f"{y:.1f}%",
            ha='center',
            va=va,
            fontsize=8,
            color='black',
            rotation=90,
            fontweight='bold'
        )



# -- Title and Labels --
plt.title("Shorts % by Creator", fontsize=18, fontweight='bold', pad=15)
plt.ylabel("Shorts %", fontsize=12)
plt.xlabel("Creator", fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=10)
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# -- Layout Fix --
plt.tight_layout()
plt.show()


In [ ]:
#Distribution of Shorts vs. Traditional by Creator, Genre, Size

# ---- Shorts ratio by creator ----
creator_counts = videos_df.groupby(['creatorName', 'is_short']).size().unstack(fill_value=0)
creator_counts['total'] = creator_counts.sum(axis=1)
creator_counts['shorts_ratio'] = creator_counts[True] / creator_counts['total']

# ---- Shorts ratio by genre ----
genre_counts = videos_df.groupby(['genre', 'is_short']).size().unstack(fill_value=0)
genre_counts['total'] = genre_counts.sum(axis=1)
genre_counts['shorts_ratio'] = genre_counts[True] / genre_counts['total']
print(genre_counts[['total', 'shorts_ratio']])

# ---- Shorts ratio by size category ----
size_counts = videos_df.groupby(['sizeCategory', 'is_short']).size().unstack(fill_value=0)
size_counts['total'] = size_counts.sum(axis=1)
size_counts['shorts_ratio'] = size_counts[True] / size_counts['total']
print(size_counts[['total', 'shorts_ratio']])

# Bar plot: % Shorts by creator
plt.figure(figsize=(14,6))
creator_counts.sort_values('shorts_ratio', ascending=False)['shorts_ratio'].plot(kind='bar', color='crimson')
plt.ylabel('% of Shorts Videos')
plt.title('Proportion of Shorts Videos by Creator')
plt.tight_layout()
plt.show()

# Visualize by genre and size category
plt.figure(figsize=(8,4))
sns.barplot(x=genre_counts.index, y=genre_counts['shorts_ratio'])
plt.title('Shorts Ratio by Genre')
plt.ylabel('% Shorts')
plt.show()

plt.figure(figsize=(8,4))
sns.barplot(x=size_counts.index, y=size_counts['shorts_ratio'])
plt.title('Shorts Ratio by Creator Size')
plt.ylabel('% Shorts')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a single figure with 2 subplots side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Shorts Ratio by Genre
sns.barplot(ax=axes[0], x=genre_counts.index, y=genre_counts['shorts_ratio'], color='steelblue')
axes[0].set_title('Shorts Ratio by Genre')
axes[0].set_ylabel('% Shorts')
axes[0].set_ylim(0, 0.65)  # extended y-axis to fit text
for i, val in enumerate(genre_counts['shorts_ratio']):
    axes[0].text(i, val + 0.015, f'{val:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=12)

# Plot 2: Shorts Ratio by Creator Size
sns.barplot(ax=axes[1], x=size_counts.index, y=size_counts['shorts_ratio'], color='steelblue')
axes[1].set_title('Shorts Ratio by Creator Size')
axes[1].set_ylabel('% Shorts')
axes[1].set_ylim(0, 0.70)  # extended y-axis to accommodate 62.6%
for i, val in enumerate(size_counts['shorts_ratio']):
    axes[1].text(i, val + 0.015, f'{val:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
#Engagement Comparison: Shorts vs. Traditional

# Boxplots of engagement for shorts vs. traditional
eng_cols = ['viewCount', 'likeCount', 'commentCount']

for col in eng_cols:
    plt.figure(figsize=(7,5))
    sns.boxplot(x='is_short', y=col, data=videos_df, showfliers=False)
    plt.yscale('log') # Engagement is highly skewed
    plt.xticks([0,1], ['Traditional', 'Shorts'])
    plt.title(f'{col} by Video Type')
    plt.ylabel(col + ' (log scale)')
    plt.show()

# Summary stats
summary = videos_df.groupby('is_short')[eng_cols].agg(['mean', 'median', 'count'])
summary.index = ['Traditional', 'Shorts']
print(summary)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Engagement columns
eng_cols = ['viewCount', 'likeCount', 'commentCount']

# Compute summary stats
summary = videos_df.groupby('is_short')[eng_cols].agg(['mean', 'median', 'count'])
summary.index = ['Traditional', 'Shorts']

# Create 1x3 subplot
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, col in enumerate(eng_cols):
    ax = axes[i]
    sns.boxplot(x='is_short', y=col, data=videos_df, showfliers=False, ax=ax)
    ax.set_yscale('log')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Traditional', 'Shorts'])
    ax.set_title(f'{col} by Video Type')
    ax.set_ylabel(f'{col} (log scale)')

    # Extract stats for this metric
    means = summary[col]['mean']
    medians = summary[col]['median']

    # Add text annotation with mean & median
    stats_text = f"Traditional:\nMean = {means[0]:,.0f}\nMedian = {medians[0]:,.0f}\n\n" \
                 f"Shorts:\nMean = {means[1]:,.0f}\nMedian = {medians[1]:,.0f}"
    ax.text(1.05, 0.5, stats_text, transform=ax.transAxes,
            fontsize=10, verticalalignment='center', bbox=dict(facecolor='white', edgecolor='gray'))

plt.tight_layout()
plt.show()


In [ ]:
# Make sure 'publishedAt' is datetime
videos_df['publishedAt'] = pd.to_datetime(videos_df['publishedAt'])

# 'posting time of each video'
videos_df['upload_hour'] = videos_df['publishedAt'].dt.hour

# Add 'year_month' for grouping
videos_df['year_month'] = videos_df['publishedAt'].dt.to_period('M')

# Label video type
videos_df['video_type'] = videos_df['is_short'].map({True: 'Shorts', False: 'Traditional'})

# Group by creator, month, and type
freq = videos_df.groupby(['creatorName', 'year_month', 'video_type']).size().reset_index(name='upload_count')

# Pivot for easy plotting
pivot = freq.pivot_table(index=['year_month', 'creatorName'], columns='video_type', values='upload_count', fill_value=0).reset_index()

# Upload Frequency Per Creator Per Month
creators = videos_df['creatorName'].unique()
n = len(creators)
fig, axes = plt.subplots(n, 1, figsize=(12, 3*n), sharex=True)
for i, creator in enumerate(creators):
    ax = axes[i] if n > 1 else axes
    creator_pivot = pivot[pivot['creatorName'] == creator].set_index('year_month')[['Shorts', 'Traditional']]
    creator_pivot.plot(kind='bar', ax=ax, stacked=False)
    ax.set_title(creator)
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
#Average Monthly Upload Frequency by Genre/Size

# Group by genre/size, month, type
grouped = videos_df.groupby(['genre', 'sizeCategory', 'year_month', 'video_type']).size().reset_index(name='count')

# Average per group (over months and creators)
genre_size_avg = grouped.groupby(['genre', 'sizeCategory', 'video_type'])['count'].mean().reset_index()

# Plot as grouped barplot
import seaborn as sns
plt.figure(figsize=(10,6))
sns.barplot(data=genre_size_avg, x='sizeCategory', y='count', hue='video_type')
plt.title('Average Monthly Uploads by Creator Size')
plt.ylabel('Avg. videos/month')
plt.xlabel('Size Category')

plt.show()

plt.figure(figsize=(8,6))
sns.barplot(data=genre_size_avg, x='genre', y='count', hue='video_type')
plt.title('Average Monthly Uploads by Genre')
plt.ylabel('Avg. videos/month')
plt.xlabel('Genre')
plt.show()


In [ ]:
# Pivot for heatmap: Uploads per Creator per Month
hm = freq.pivot_table(index='creatorName', columns=['year_month','video_type'], values='upload_count', fill_value=0)
plt.figure(figsize=(16,10))
sns.heatmap(hm, annot=False, cmap='Blues')
plt.title('Uploads per Creator per Month (by Type)')
plt.ylabel('Creator')
plt.xlabel('Month and Video Type')
plt.show()


In [ ]:
#Monthly Avg Upload Rate Table

# Group by creator, video_type, and year_month, then average per creator
monthly_freq = videos_df.groupby(['creatorName', 'video_type', 'year_month']).size().reset_index(name='num_videos')
avg_monthly = monthly_freq.groupby(['creatorName', 'video_type'])['num_videos'].mean().unstack().fillna(0)
avg_monthly['Total'] = avg_monthly.sum(axis=1)
avg_monthly['Shorts_Ratio'] = avg_monthly.get('Shorts', 0) / avg_monthly['Total']
print(avg_monthly.sort_values('Shorts_Ratio', ascending=False))


In [ ]:
# Convert channel creation date to datetime (ensure UTC)
channels_df['channelCreationDate'] = pd.to_datetime(channels_df['channelCreationDate'], errors='coerce', utc=True)
DATA_PULL_DATE = pd.to_datetime('2025-07-26').tz_localize('UTC')
channels_df['channel_age_years'] = (DATA_PULL_DATE - channels_df['channelCreationDate']).dt.days / 365.25

channels_df = channels_df.fillna({'subscriberCount': 0, 'videoCount': 0, 'videosInLastYear': 0, 'viewCount': 0})

# Only consider videos from the last 6 months (relative to scraping date)
videos_df['publishedAt'] = pd.to_datetime(videos_df['publishedAt'], errors='coerce', utc=True)
cutoff = DATA_PULL_DATE - pd.DateOffset(months=12)
recent_videos = videos_df[videos_df['publishedAt'] >= cutoff]

# Count Shorts and Traditional per channel
shorts_counts = recent_videos.groupby('channelId')['is_short'].value_counts().unstack().fillna(0)
if True not in shorts_counts.columns: shorts_counts[True] = 0
if False not in shorts_counts.columns: shorts_counts[False] = 0
shorts_counts = shorts_counts.rename(columns={False: 'traditional_videos_12mo', True: 'shorts_videos_12mo'})
for col in ['traditional_videos_12mo', 'shorts_videos_12mo']:
    if col not in shorts_counts.columns:
        shorts_counts[col] = 0

channels_df = channels_df.merge(shorts_counts[['traditional_videos_12mo', 'shorts_videos_12mo']],
                               left_on='channelId', right_index=True, how='left').fillna(0)

# Calculate proportions and rates
channels_df['total_videos_12mo'] = channels_df['traditional_videos_12mo'] + channels_df['shorts_videos_12mo']
channels_df['pct_traditional_12mo'] = channels_df['traditional_videos_12mo'] / channels_df['total_videos_12mo'].replace(0, np.nan)
channels_df['pct_shorts_12mo'] = channels_df['shorts_videos_12mo'] / channels_df['total_videos_12mo'].replace(0, np.nan)

channels_df['uploads_per_month_traditional'] = channels_df['traditional_videos_12mo'] / 12
channels_df['pct_recent_uploads_traditional'] = channels_df['traditional_videos_12mo'] / channels_df['videoCount'].replace(0, np.nan)
channels_df['avg_views_per_video'] = channels_df['viewCount'] / channels_df['videoCount'].replace(0, np.nan)

# Avg views per traditional video (last 12mo)
trad_vids_12mo = recent_videos[recent_videos['is_short'] == False]
trad_views = trad_vids_12mo.groupby('channelId')['viewCount'].mean().rename('avg_views_traditional_12mo')
channels_df = channels_df.merge(trad_views, on='channelId', how='left')

# Log transforms for skewed features
for col in ['subscriberCount', 'videoCount', 'viewCount', 'traditional_videos_12mo', 'shorts_videos_12mo']:
    channels_df[f'log_{col}'] = np.log1p(channels_df[col])

# --- Tidy up: Fill any new NaNs created by division by zero ---
channels_df = channels_df.fillna(0)


channels_df

In [ ]:
# How many videos per creator
# Barplot of videos in last 6 months per creator
plt.figure(figsize=(12,5))
channels_df = channels_df.sort_values('traditional_videos_12mo', ascending=False)
plt.bar(channels_df['name'], channels_df['traditional_videos_12mo'], color='royalblue')
plt.xticks(rotation=80, ha='right', fontsize=9)
plt.xlabel('Creator Name')
plt.ylabel('Traditional Videos in Last year')
plt.title('Videos Collected per Creator')
plt.tight_layout()
plt.show()

# Print basic stats
print('Summary statistics for videos in last year per creator:')
print(channels_df['traditional_videos_12mo'].describe())
print('\nCreators with fewer than 20 videos:')
print(channels_df[channels_df['traditional_videos_12mo'] < 20][['name', 'traditional_videos_12mo']])



# How many videos per creator
# Barplot of videos in last 6 months per creator
plt.figure(figsize=(12,5))
channels_df = channels_df.sort_values('uploads_per_month_traditional', ascending=False)
plt.bar(channels_df['name'], channels_df['uploads_per_month_traditional'], color='royalblue')
plt.xticks(rotation=80, ha='right', fontsize=9)
plt.xlabel('Creator Name')
plt.ylabel('Traditional Videos uploads per Month')
plt.title('Videos uploaded per Creator per month')
plt.tight_layout()
plt.show()

# Print basic stats
print('Summary statistics for traditional videos uploaded per creator per month:')
print(channels_df['uploads_per_month_traditional'].describe())
print('\nCreators with fewer than 20 videos:')
print(channels_df[channels_df['uploads_per_month_traditional'] < 20][['name', 'uploads_per_month_traditional']])



In [ ]:
channels_df.info()
channels_df.describe().T

In [ ]:
#Filtering only traditional videos and comments for further analysis

traditional_videos_df = videos_df[~videos_df['is_short']].copy()
traditional_videos_df = traditional_videos_df.drop(['tags', 'hashtags', 'categoryId', 'videoUrl', 'is_short', 'video_type'], axis=1)
traditional_videos_df

In [ ]:
traditional_videos_df.info()
traditional_videos_df.describe()

In [ ]:
# Overall engagement distribution: view, likes, comments

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fields = ['viewCount', 'likeCount', 'commentCount']
titles = ['Views per Video', 'Likes per Video', 'Comments per Video']

for i, field in enumerate(fields):
    values = traditional_videos_df[field].clip(lower=1) # Avoid log(0)
    axes[i].hist(np.log10(values), bins=40, color='teal', edgecolor='k', alpha=0.7)
    axes[i].set_xlabel(f'log10({field})')
    axes[i].set_ylabel('Number of Videos')
    axes[i].set_title(titles[i])

plt.tight_layout()
plt.show()

# Print summary stats for each metric
for field in fields:
    print(f"\nSummary statistics for {field}:")
    print(traditional_videos_df[field].describe())

In [ ]:
# Per video Engagement Averages by Creator

# Group by creator
by_creator = traditional_videos_df.groupby('creatorName').agg(
    num_videos=('videoId', 'count'),
    avg_views=('viewCount', 'mean'),
    avg_likes=('likeCount', 'mean'),
    avg_comments=('commentCount', 'mean')
).sort_values('avg_views', ascending=False)

print("\nTop 10 creators by average video views:")
print(by_creator.head(10)[['avg_views', 'avg_likes', 'avg_comments']])
print("\nBottom 5 creators by average video views:")
print(by_creator.tail(5)[['avg_views', 'avg_likes', 'avg_comments']])

# Optionally, plot
by_creator[['avg_views', 'avg_likes', 'avg_comments']].plot(kind='bar', figsize=(14,6))
plt.ylabel('Average Per Video')
plt.title('Average Per-Video Engagement by Creator')
plt.xticks(rotation=80, ha='right', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Show top 10 most viewed videos
top_videos = traditional_videos_df.sort_values('viewCount', ascending=False).head(10)
print("\nTop 10 most viewed videos in dataset:")
print(top_videos[['creatorName', 'title', 'viewCount', 'likeCount', 'commentCount']])